# 📰 GDELT Full Text Search — Historical Data (2017–2022)

## Instructions
1. Change `START_DATE` and `END_DATE` in **Cell 1**
2. Run **Runtime → Run All** — everything is automatic
3. Checkpoint saved after **every topic** — if Colab disconnects, re-run and it resumes automatically
4. Checkpoint downloaded automatically every 2 topics as backup
5. Final CSV downloaded automatically when all topics finish

## Suggested date chunks per session
```
Session 1 : START=2017,1,1   END=2017,6,30
Session 2 : START=2017,7,1   END=2017,12,31
Session 3 : START=2018,1,1   END=2018,12,31
Session 4 : START=2019,1,1   END=2019,12,31
Session 5 : START=2020,1,1   END=2020,12,31
Session 6 : START=2021,1,1   END=2021,12,31
Session 7 : START=2022,1,1   END=2022,12,31
```
After all sessions → run the **Consolidation notebook** to merge everything.

In [ ]:
# ═══════════════════════════════════════════════════════
# CHANGE ONLY THESE TWO LINES EACH SESSION
START_DATE_ARGS = (2017, 1, 1)    # ← year, month, day
END_DATE_ARGS   = (2017, 6, 30)   # ← year, month, day
# ═══════════════════════════════════════════════════════

# ── Install ───────────────────────────────────────────
import subprocess
subprocess.run(['pip', 'install', 'requests', 'pandas', '-q'], check=True)

# ── Imports ───────────────────────────────────────────
import requests
import pandas as pd
import numpy as np
import time
import os
import threading
from datetime import datetime, timedelta
from google.colab import files

START_DATE = datetime(*START_DATE_ARGS)
END_DATE   = datetime(*END_DATE_ARGS)

# ── Settings ──────────────────────────────────────────
GDELT_BASE       = 'https://api.gdeltproject.org/api/v2/doc/doc'
MAX_RECORDS      = 250
CHUNK_DAYS       = 60      # 60-day chunks per API call
SLEEP_CHUNK      = 10.0    # 10s between chunks
SLEEP_TOPIC      = 12.0    # 12s between topics
SLEEP_429        = 90.0    # 90s wait on rate limit
CHECKPOINT_FILE  = f'checkpoint_{START_DATE.strftime("%Y%m%d")}_{END_DATE.strftime("%Y%m%d")}.csv'
OUTPUT_FILE      = f'gdelt_fts_{START_DATE.strftime("%Y%m%d")}_{END_DATE.strftime("%Y%m%d")}.csv'

TOPICS = {
    'NIFTY_50':            'Nifty India stock market',
    'RBI_REPO_RATE':       'Reserve Bank India repo rate policy',
    'USD_INR':             'dollar rupee exchange rate India currency',
    'CRUDE_OIL':           'Brent crude oil price India energy',
    'US_FED':              'Federal Reserve interest rate policy meeting',
    'FII_DII_FLOWS':       'foreign institutional investor India inflows',
    'GLOBAL_MARKETS':      'global stock markets Wall Street rally',
    'INDIA_CPI_INFLATION': 'India inflation consumer price index',
    'RELIANCE_INDUSTRIES': 'Reliance Industries stock market India',
    'HDFC_BANK':           'HDFC Bank India quarterly results',
}

print('=' * 60)
print('  GDELT FULL TEXT SEARCH')
print('=' * 60)
print(f'  Date range  : {START_DATE.date()} → {END_DATE.date()}')
print(f'  Topics      : {len(TOPICS)}')
print(f'  Checkpoint  : {CHECKPOINT_FILE}')
print(f'  Output      : {OUTPUT_FILE}')

# ── Keep-alive thread ─────────────────────────────────
# Prevents Colab from disconnecting due to inactivity
_keepalive = True
def keepalive():
    count = 0
    while _keepalive:
        time.sleep(60)
        count += 1
        if count % 5 == 0:
            print(f'  ⏱️  Still running... ({count} minutes elapsed)', flush=True)
t = threading.Thread(target=keepalive, daemon=True)
t.start()
print('  ✓ Keep-alive started')

# ── Fetch function ────────────────────────────────────
def fetch_topic(topic_name, query, start_dt, end_dt):
    all_articles  = []
    current_start = start_dt
    while current_start < end_dt:
        current_end = min(current_start + timedelta(days=CHUNK_DAYS), end_dt)
        params = {
            'query':         f'{query} sourcelang:english',
            'mode':          'artlist',
            'maxrecords':    MAX_RECORDS,
            'startdatetime': current_start.strftime('%Y%m%d%H%M%S'),
            'enddatetime':   current_end.strftime('%Y%m%d%H%M%S'),
            'format':        'json',
            'sort':          'DateDesc',
        }
        retry = 0
        while retry < 3:
            try:
                resp = requests.get(GDELT_BASE, params=params, timeout=30)
                if resp.status_code == 429:
                    print(f'    ⚠️  Rate limit — waiting {SLEEP_429:.0f}s  (retry {retry+1}/3)')
                    time.sleep(SLEEP_429)
                    retry += 1
                    continue
                if len(resp.text.strip()) == 0:
                    print(f'    ⚠️  Empty response — retry {retry+1}/3')
                    time.sleep(8)
                    retry += 1
                    continue
                if resp.status_code != 200:
                    print(f'    ⚠️  HTTP {resp.status_code} — skipping')
                    break
                if 'Invalid query' in resp.text or 'too short' in resp.text:
                    print(f'    ⚠️  Query rejected: {resp.text[:60]}')
                    break
                try:
                    data = resp.json()
                except Exception:
                    print(f'    ⚠️  JSON error — retry {retry+1}/3')
                    time.sleep(8)
                    retry += 1
                    continue
                articles = data.get('articles', [])
                for a in articles:
                    all_articles.append({
                        'topic':        topic_name,
                        'published_at': a.get('seendate', ''),
                        'source':       a.get('domain', ''),
                        'title':        a.get('title', ''),
                        'url':          a.get('url', ''),
                        'language':     a.get('language', ''),
                        'tone':         a.get('tone', None),
                        'country':      a.get('sourcecountry', ''),
                    })
                label = f"{current_start.strftime('%d-%b-%Y')}–{current_end.strftime('%d-%b-%Y')}"
                print(f'    {label}: {len(articles):>3} articles  (total: {len(all_articles)})')
                break
            except requests.exceptions.Timeout:
                print(f'    ⚠️  Timeout — retry {retry+1}/3')
                retry += 1
                time.sleep(5)
            except Exception as e:
                print(f'    ⚠️  {str(e)[:60]} — retry {retry+1}/3')
                retry += 1
                time.sleep(5)
        current_start = current_end
        time.sleep(SLEEP_CHUNK)
    return all_articles

# ── Load checkpoint ───────────────────────────────────
all_articles     = []
completed_topics = set()

if os.path.exists(CHECKPOINT_FILE):
    ckpt_df = pd.read_csv(CHECKPOINT_FILE)
    all_articles     = ckpt_df.to_dict('records')
    completed_topics = set(ckpt_df['topic'].unique())
    print(f'\n✅ Checkpoint found — resuming from {len(completed_topics)} completed topics')
    print(f'   Done     : {completed_topics}')
    print(f'   Articles : {len(all_articles)}')
else:
    print('\n  No checkpoint — starting fresh')
    print('  Waiting 10s before first request...')
    time.sleep(10)

# ── Main fetch loop ───────────────────────────────────
topic_summary = {}
print()
print('=' * 60)
print(f'  FETCHING — {START_DATE.date()} → {END_DATE.date()}')
print('=' * 60)

for idx, (topic_name, query) in enumerate(TOPICS.items(), 1):

    if topic_name in completed_topics:
        existing = sum(1 for a in all_articles if a['topic'] == topic_name)
        topic_summary[topic_name] = existing
        print(f'\n[{idx:>2}/{len(TOPICS)}] {topic_name} — ✅ already done ({existing} articles)')
        continue

    print(f'\n[{idx:>2}/{len(TOPICS)}] {topic_name}')
    print(f'  Query: {query}')
    articles = fetch_topic(topic_name, query, START_DATE, END_DATE)
    topic_summary[topic_name] = len(articles)

    if articles:
        all_articles.extend(articles)
        print(f'  ✓ {len(articles)} articles')
    else:
        print(f'  ⚠️  0 articles')

    # Save checkpoint after every topic
    ckpt_df = pd.DataFrame(all_articles)
    ckpt_df.to_csv(CHECKPOINT_FILE, index=False)
    print(f'  💾 Checkpoint saved ({len(all_articles)} total articles)')

    # Auto-download checkpoint every 2 topics as backup
    if idx % 2 == 0:
        try:
            files.download(CHECKPOINT_FILE)
            print(f'  ⬇️  Checkpoint downloaded as backup')
        except Exception:
            pass

    time.sleep(SLEEP_TOPIC)

# ── Stop keep-alive ───────────────────────────────────
_keepalive = False

# ── Save final output ─────────────────────────────────
print()
print('=' * 60)
print('  COMPLETE')
print('=' * 60)

if all_articles:
    final_df = pd.DataFrame(all_articles)
    final_df['published_at'] = pd.to_datetime(
        final_df['published_at'], format='%Y%m%dT%H%M%SZ', errors='coerce'
    )
    final_df['date'] = final_df['published_at'].dt.date
    final_df['tone'] = pd.to_numeric(final_df['tone'], errors='coerce')
    final_df = final_df.drop_duplicates(subset=['url','topic'])
    final_df = final_df.sort_values('published_at').reset_index(drop=True)
    final_df.to_csv(OUTPUT_FILE, index=False)

    print(f'  Total articles : {len(final_df)}')
    print(f'  Date range     : {final_df["date"].min()} → {final_df["date"].max()}')
    print(f'  Output file    : {OUTPUT_FILE}')
    print()
    print('  Articles per topic:')
    for t, c in sorted(topic_summary.items(), key=lambda x: x[1], reverse=True):
        bar = '█' * min(c // 20, 25)
        print(f'  {t:<30} : {c:>4}  {bar}')
    print()
    print('⬇️  Downloading final file...')
    files.download(OUTPUT_FILE)
    print(f'  ✓ {OUTPUT_FILE} downloaded')
    print()
    print('✅ Session complete!')
    print(f'  Next session → change START_DATE_ARGS and END_DATE_ARGS at top of this cell')
    next_start = END_DATE + timedelta(days=1)
    next_end   = min(next_start.replace(month=12, day=31), datetime(2022, 12, 31))
    print(f'  Suggested next: START=({next_start.year},{next_start.month},{next_start.day})  END=({next_end.year},{next_end.month},{next_end.day})')
else:
    print('  ⚠️  No articles collected')


In [ ]:
# ═══════════════════════════════════════════════════════
# CONSOLIDATION — Run this ONCE after all sessions done
# Upload all session files + step5a_raw_news_all_topics.csv
# ═══════════════════════════════════════════════════════
import subprocess
subprocess.run(['pip', 'install', 'pandas', '-q'], check=True)

import pandas as pd
import os
from google.colab import files

print('Upload ALL files — all session CSVs + existing step5a_raw_news_all_topics.csv')
uploaded = files.upload()
print(f'\n  Files uploaded: {len(uploaded)}')

dfs = []
for fname in uploaded.keys():
    try:
        df = pd.read_csv(fname)
        dfs.append(df)
        print(f'  ✓ {fname}: {len(df):,} rows')
    except Exception as e:
        print(f'  ⚠️  Error loading {fname}: {e}')

if dfs:
    master = pd.concat(dfs, ignore_index=True)
    master['published_at'] = pd.to_datetime(master['published_at'], errors='coerce')
    master['date']         = master['published_at'].dt.date
    master['tone']         = pd.to_numeric(master['tone'], errors='coerce')
    before = len(master)
    master = master.drop_duplicates(subset=['url','topic'])
    master = master.sort_values('published_at').reset_index(drop=True)

    MASTER = 'step5a_raw_news_all_topics.csv'
    master.to_csv(MASTER, index=False)
    size = os.path.getsize(MASTER)

    print()
    print('=' * 60)
    print('  MASTER FILE')
    print('=' * 60)
    print(f'  Rows          : {len(master):,}  (removed {before-len(master):,} duplicates)')
    print(f'  Date range    : {master["date"].min()} → {master["date"].max()}')
    print(f'  Unique dates  : {master["date"].nunique():,}')
    print(f'  File size     : {size/1024/1024:.1f} MB')
    print()
    master['year'] = master['published_at'].dt.year
    print('  Coverage by year:')
    for yr, cnt in master.groupby('year').size().sort_index().items():
        bar = '█' * min(cnt // 200, 20)
        print(f'    {yr}: {cnt:>6,}  {bar}')
    print()
    print('⬇️  Downloading...')
    files.download(MASTER)
    print(f'  ✓ {MASTER} downloaded')
    print('✅ Consolidation complete!')
